## 使用说明

### 标准流程（Colab 跑 3d_ff）
1. Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5 → Cell 6

### 备用流程（Colab 跑 2d，本机显存不足时）
1. Cell 1 → Cell 2 → Cell 2.5 → Cell 3 → Cell 4 → Cell 7 → Cell 6

### 注意事项
- Cell 2 会自动切换到旧版本（b5bcd5d），包含内置 utils3d
- Cell 2.5 切换回新版本，用于 2d 备用方案
- 每次断联重启后需重新运行 Cell 2
- bremm.png 如损坏需手动上传（本机路径见 Cell 2 输出）

# Track4World — Colab 推理环境
**GPU**: T4 16GB (免费版) 或 A100 (Colab Pro)

**用途**: 在本机显存不足时，使用 Colab 跑大视频推理，结果下载回本地

## 使用流程
1. 运行 Cell 1：检查 GPU
2. 运行 Cell 2：克隆仓库 + 安装依赖（首次约15分钟）
3. 运行 Cell 3：下载预训练权重
4. 运行 Cell 4：上传视频文件
5. 运行 Cell 5：执行推理
6. 运行 Cell 6：下载结果

In [ ]:
# Cell 1: 检查 GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
!nvidia-smi

In [ ]:
# Cell 2: 克隆 Track4World + 安装依赖 + 修复 utils3d
import os, sys

# ── 检测是否已完成安装（重启后跳过安装步骤）──
_flag = "/content/.t4w_installed"

if not os.path.exists(_flag):
    # ── 第一次运行：克隆 + 安装 ──
    if not os.path.exists("/content/Track4World"):
        !git clone --recurse-submodules https://github.com/TencentARC/Track4World.git /content/Track4World
    else:
        !git -C /content/Track4World submodule update --init --recursive

    os.chdir("/content/Track4World")
    !git reset --hard b5bcd5d
    print("已切换到 b5bcd5d")

    # 修复 pi3 软链接（仅当 pi3x 存在时）
    pi3_dst = "/content/Track4World/track4world/nets/external/pi3"
    pi3_src = "/content/Track4World/track4world/nets/external/pi3x"
    if os.path.islink(pi3_dst): os.unlink(pi3_dst)
    if os.path.exists(pi3_src) and not os.path.exists(pi3_dst):
        os.symlink(pi3_src, pi3_dst); print("已创建 pi3 软链接")
    else:
        print("[skip] pi3x 不存在，跳过软链接")

    # 先固定 numpy，避免后续包触发版本冲突
    !pip install -q --force-reinstall numpy==1.26.4

    # 修改 requirements.txt，放宽 open3d 版本限制
    !sed -i 's/open3d==0.18.0/open3d>=0.19.0/' /content/Track4World/requirements.txt

    # 安装其余依赖
    !pip install -q -r /content/Track4World/requirements.txt || true
    !pip install -q open3d viser tqdm matplotlib plotly
    !pip uninstall -y -q utils3d || true
    !pip install -q --force-reinstall --no-deps opencv-python

    # 写入标志文件，重启后跳过安装
    open(_flag, "w").close()
    print("\n依赖安装完成，正在重启运行时以使 numpy 版本生效...")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

else:
    # ── 重启后：配置路径 + 补装缺失包 + 验证 ──
    os.chdir("/content/Track4World")
    sys.path.insert(0, '/content/Track4World')
    for mod in list(sys.modules.keys()):
        if mod.startswith('utils3d'):
            del sys.modules[mod]

    # 补装可能在重启后丢失的包
    try:
        import open3d
    except ImportError:
        print("open3d 未找到，正在安装...")
        !pip install -q open3d
        print("open3d 安装完成")

    print("=== 关键依赖检查 ===")
    missing = []
    for pkg, mod in [
        ('torch','torch'), ('numpy','numpy'), ('cv2','cv2'),
        ('open3d','open3d'), ('einops','einops'), ('timm','timm'),
        ('viser','viser'), ('plotly','plotly'),
    ]:
        try:
            m = __import__(mod)
            print(f"  [ok] {pkg}: {getattr(m,'__version__','?')}")
        except ImportError:
            print(f"  [!!] {pkg}: 未安装")
            missing.append(pkg)

    # utils3d 单独检查（内置版本，不通过 pip 安装）
    try:
        import utils3d
        has_func = hasattr(utils3d.torch, 'depth_to_points')
        print(f"  [ok] utils3d (内置): depth_to_points={has_func}")
        if not has_func:
            missing.append('utils3d.depth_to_points')
    except ImportError:
        print("  [!!] utils3d: 导入失败")
        missing.append('utils3d')

    if missing:
        print(f"\n[!] 依赖问题: {missing}")
    else:
        print("\n所有依赖就绪，可继续运行 Cell 3")

In [ ]:
# Cell 3: 下载预训练权重（MOGE + DA3）
import os
os.makedirs("/content/Track4World/checkpoints", exist_ok=True)

# 下载MOGE权重（默认）
moge_path = "/content/Track4World/checkpoints/track4world_moge.pth"
if not os.path.exists(moge_path):
    !wget -q --show-progress -O {moge_path} https://huggingface.co/TencentARC/Track4World/resolve/main/track4world_moge.pth
else:
    print("MOGE权重已存在")

# 下载DA3权重（改善点云覆盖率）
da3_path = "/content/Track4World/checkpoints/track4world_da3.pth"
if not os.path.exists(da3_path):
    print("\n下载DA3权重（改善点云覆盖率）...")
    !wget -q --show-progress -O {da3_path} https://huggingface.co/TencentARC/Track4World/resolve/main/track4world_da3.pth
else:
    print("DA3权重已存在")

print(f"\nMOGE权重: {os.path.getsize(moge_path)/1e6:.1f} MB")
print(f"DA3权重: {os.path.getsize(da3_path)/1e6:.1f} MB")
print("\n推荐：使用DA3可改善点云覆盖率（从47%提升到60%+）")

In [ ]:
# Cell 4: 上传视频
from google.colab import files
import os

os.makedirs("/content/Track4World/input_videos", exist_ok=True)
print("请选择要上传的视频文件 (.mp4)")
uploaded = files.upload()

for fname in uploaded:
    dst = f"/content/Track4World/input_videos/{fname}"
    with open(dst, "wb") as f:
        f.write(uploaded[fname])
    print(f"已保存: {dst} ({os.path.getsize(dst)/1e6:.1f} MB)")

VIDEO_PATH = f"/content/Track4World/input_videos/{list(uploaded.keys())[0]}"
print(f"将使用视频: {VIDEO_PATH}")

In [ ]:
# Cell 4.5: 生成运动区域 mask（可选，本次测试不使用）
# ⚠️ 本次测试不使用mask，跳过此cell
# 如需使用mask，取消注释以下代码

"""
import cv2
import numpy as np
from tqdm.auto import tqdm
import os

MASK_DIR = "/content/Track4World/results/output_3d_efep/mask"
os.makedirs(MASK_DIR, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"视频总帧数: {total_frames}")

# 改进参数：更低阈值 + 更大 close 核
bg_subtractor = cv2.createBackgroundSubtractorMOG2(
    history=200,
    varThreshold=8,
    detectShadows=False
)

kernel_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))

frame_idx = 0
with tqdm(total=total_frames, desc="生成 mask") as pbar:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        fg_mask = bg_subtractor.apply(frame, learningRate=-1)
        fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN,  kernel_open)
        fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, kernel_close)
        _, fg_mask = cv2.threshold(fg_mask, 127, 255, cv2.THRESH_BINARY)

        cv2.imwrite(f"{MASK_DIR}/mask_{frame_idx:04d}.png", fg_mask)
        frame_idx += 1
        pbar.update(1)

cap.release()

# 验证效果
for i in [10, 50, 100]:
    m = cv2.imread(f"{MASK_DIR}/mask_{i:04d}.png", cv2.IMREAD_GRAYSCALE)
    ratio = (m > 127).sum() / m.size
    print(f"mask_{i:04d}: dynamic_ratio={ratio:.4f}")

print(f"完成！生成了 {frame_idx} 个 mask 文件到 {MASK_DIR}")
"""

print("⚠️ 本次测试不使用mask，已跳过mask生成")

In [ ]:
# Cell 5: 3d_efep 推理（推荐使用MOGE，不使用mask）
# ============ 可修改参数 ============
IMAGE_SIZE = 448
MAX_FRAMES = 20  # 测试用20帧
USE_DA3 = False  # 使用MOGE（推荐，避免坐标系问题）
# ====================================

import os
os.chdir("/content/Track4World")
os.environ["PYTHONIOENCODING"] = "utf-8"

# ⚠️ 重要：删除mask文件夹，避免使用旧mask
mask_dir = "/content/Track4World/results/output_3d_efep/mask"
if os.path.exists(mask_dir):
    import shutil
    shutil.rmtree(mask_dir)
    print(f"✓ 已删除旧mask: {mask_dir}")

# 选择权重和坐标系
if USE_DA3:
    ckpt = "checkpoints/track4world_da3.pth"
    coord = "world_depthanythingv3"
    print("⚠️ 使用DA3模型（改善点云覆盖率）")
    print("   注意：DA3使用world坐标系，可能需要后处理修正Z轴方向")
else:
    ckpt = "checkpoints/track4world_moge.pth"
    coord = "camera_base"
    print("✓ 使用MOGE模型（默认，与本地环境一致）")

cmd = (
    f"python demo.py"
    f" --mp4_path {VIDEO_PATH}"
    f" --mode 3d_efep"
    f" --coordinate {coord}"
    f" --ckpt_init {ckpt}"
    f" --image_size {IMAGE_SIZE}"
    f" --max_frames {MAX_FRAMES}"
    f" --save_base_dir /content/Track4World/results/output_3d_efep"
)
print(f"执行命令:\n{cmd}\n")
!{cmd}
print("\n✓ 3d_efep 推理完成")

In [ ]:
# Cell 6: 打包并下载结果
import os, glob
from google.colab import files

os.chdir("/content/Track4World")

# 查找结果目录（支持多种路径）
result_dirs = sorted(glob.glob("results/*/"), key=os.path.getmtime, reverse=True)
result_dirs += sorted(glob.glob("results/output_*/"), key=os.path.getmtime, reverse=True)

if not result_dirs:
    print("未找到结果目录，请先运行 Cell 5 或 Cell 7")
else:
    latest = result_dirs[0].rstrip("/")
    print(f"打包结果目录: {latest}")
    zip_name = f"/content/{latest.replace('/', '_')}_results.zip"
    !zip -r {zip_name} {latest}
    print(f"压缩包大小: {os.path.getsize(zip_name)/1e6:.1f} MB")
    files.download(zip_name)
    print("下载已启动，请查看浏览器下载列表")

In [ ]:
# Cell 2.5: 切换到新版本（仅用于 2d 模式备用方案）
# ⚠️ 只在本机无法跑 2d 时使用，正常情况跳过此 cell
import os
os.chdir("/content/Track4World")

# 切换回 master 分支
!git reset --hard origin/master
print("✓ 已切换到 master 分支（新版本）")

# 重新安装外部 utils3d
!pip install -q git+https://github.com/EasternJournalist/utils3d.git
print("✓ 已安装外部 utils3d（新版本不需要 depth_to_points）")

In [ ]:
# Cell 7: 2d 推理（备用方案，需先运行 Cell 2.5）
# ⚠️ 仅在本机无法跑 2d 时使用
# ============ 可修改参数 ============
IMAGE_SIZE = 320
MAX_FRAMES = 30
# ====================================

import os
os.chdir("/content/Track4World")
os.environ["PYTHONIOENCODING"] = "utf-8"

cmd = (
    f"python demo.py"
    f" --mp4_path {VIDEO_PATH}"
    f" --mode 2d"
    f" --coordinate camera_base"
    f" --ckpt_init checkpoints/track4world_moge.pth"
    f" --image_size {IMAGE_SIZE}"
    f" --max_frames {MAX_FRAMES}"
)
print(f"执行命令:\\n{cmd}\\n")
!{cmd}
print("\\n✓ 2d 推理完成")

## 参数建议（T4 16GB vs 本机 4GB）

| 模式 | image_size | max_frames | T4预计耗时 |
|------|-----------|------------|----------|
| 2d | 448 | 50 | ~3 min |
| 2d | 512 | 30 | ~3 min |
| 3d_ff | 448 | 30 | ~5 min |
| 3d_efep | 448 | 20 | ~6 min |

## Colab 免费版注意事项
- 每天约 ~4小时 GPU 使用时间
- 断开后文件全部丢失，**及时运行 Cell 6 下载结果**
- 第一次安装依赖约需 10~15 分钟，断开重连需重新安装
- 如需更长时间：Colab Pro (0/月) 或 AutoDL 按需计费